In [0]:
# ── 0. Setup ───────────────────────────────────────────────────────────────
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

CATALOG = dbutils.widgets.get("catalog")

BRONZE = f"{CATALOG}.skybound_bronze.weather_reports_bronze"
SILVER = f"{CATALOG}.skybound_silver.weather_reports_silver"

CHECKPOINT_PATH = f"/Volumes/{CATALOG}/skybound_bronze/checkpoints/weather_to_silver"

In [0]:
%skip

decode_wx structure and abbreviations:

[-/+]  [VC]  [descriptor]  [phenomenon]

intensity prefix:  - = light    (no prefix) = moderate    + = heavy
proximity:         VC = in vicinity (within 8km but not at station)
descriptor:        SH = shower    TS = thunderstorm    FZ = freezing
                   BL = blowing   DR = drifting        MI = shallow
                   PR = partial   BC = patches
phenomenon:        RA = rain      SN = snow    GS = small hail
                   GR = hail      FG = fog     BR = mist
                   HZ = haze      DZ = drizzle FU = smoke
                   SA = sand      DU = dust    VA = volcanic ash
                   SQ = squall    FC = funnel cloud/tornado
                   PO = dust whirl SS = sandstorm DS = duststorm

EXAMPLES: -RA -SN; -RA; VCSH; -SHRA; -RA -DZ, -SN; SHGS SHRA; -SHRA -SHSN; -SHSN; -TSRA; RA BR; 

In [0]:
# ── 1. Weather code decoder  ────────────────────────────────────
INTENSITIES  = {"-": "Light", "+": "Heavy"}
PROXIMITIES  = {"VC": "In vicinity"}
DESCRIPTORS  = {
    "TS": "Thunderstorm", "FZ": "Freezing",  "SH": "Shower",
    "BL": "Blowing", "DR": "Drifting",  "BC": "Patches of",
    "PR": "Partial", "MI": "Shallow",
}
PHENOMENA = {
    "GR": "Hail", "GS": "Small hail", "PL": "Ice pellets",
    "SN": "Snow", "SG": "Snow grains", "IC": "Ice crystals",
    "RA": "Rain", "DZ": "Drizzle", "UP": "Unknown precipitation",
    "FG": "Fog", "BR": "Mist", "HZ": "Haze",
    "FU": "Smoke", "VA": "Volcanic ash", "DU": "Dust",
    "SA": "Sand", "FC": "Funnel cloud", "SQ": "Squall",
    "PO": "Dust whirl", "SS": "Sandstorm", "DS": "Duststorm",
}

def decode_wx_group(group):
    rem = group
    parts = []
    if rem and rem[0] in INTENSITIES:
        parts.append(INTENSITIES[rem[0]])
        rem = rem[1:]
    vc_suffix = None
    if rem.startswith("VC"):
        vc_suffix = "In vicinity"
        rem = rem[2:]
    for code, name in DESCRIPTORS.items():
        if rem.startswith(code):
            parts.append(name)
            rem = rem[len(code):]
            break
    if rem in PHENOMENA:
        parts.append(PHENOMENA[rem])
    elif rem:
        parts.append(rem)
    if vc_suffix:
        parts.append(vc_suffix)
    return " ".join(parts) if parts else group

def decode_wx_string(wx):
    if not wx or wx in ("null", ""):
        return None
    return ", ".join(decode_wx_group(g) for g in wx.strip().split())

decode_wx_udf = F.udf(decode_wx_string, StringType())


In [0]:
# ── 2. Transform function (applied per micro-batch) ────────────────────────
def transform_and_merge(batch_df, batch_id):

    from delta.tables import DeltaTable

    if batch_df.isEmpty():
        return

    transformed = (
        batch_df
        .filter(F.col("obsTime").isNotNull())
        .filter(F.col("icaoId").isNotNull())
                
        .withColumn("obs_time",            F.to_timestamp(F.col("obsTime").cast("long")))
        .withColumn("receipt_time",        F.to_timestamp("receiptTime"))
        .withColumn("temp_c",              F.col("temp").cast("double"))
        .withColumn("dewpoint_c",          F.col("dewp").cast("double"))
        .withColumn("wind_dir_variable",   F.col("wdir") == "VRB")
        .withColumn("wind_dir_degrees",
            F.when(F.col("wdir") == "VRB", F.lit(None))
             .otherwise(F.col("wdir").cast("int")))
        .withColumn("wind_speed_kt",       F.col("wspd").cast("int"))
        .withColumn("wind_gust_kt",        F.col("wgst").cast("int"))
        .withColumn("visibility_unlimited", F.col("visib") == "6+")
        .withColumn("visibility_sm",
            F.when(F.col("visib") == "6+", F.lit(6.0))
             .otherwise(F.col("visib").cast("double")))
        .withColumn("altim_hpa",           F.col("altim").cast("double"))
        .withColumn("cloud_base_ft",       F.col("cloud_base_ft").try_cast("int"))
        .withColumn("latitude",            F.col("lat").cast("double"))
        .withColumn("longitude",           F.col("lon").cast("double"))
        .withColumn("elev_m",              F.col("elev").cast("int"))
        .withColumn("wx_string_decoded",   decode_wx_udf(F.col("wxString")))
        .withColumn("cover_description",
            F.when(F.col("cover") == "CAVOK", "Ceiling and visibility OK")
             .when(F.col("cover") == "CLR",   "Clear")
             .when(F.col("cover") == "FEW",   "Few clouds (1-2 oktas)")
             .when(F.col("cover") == "SCT",   "Scattered clouds (3-4 oktas)")
             .when(F.col("cover") == "BKN",   "Broken clouds (5-7 oktas)")
             .when(F.col("cover") == "OVC",   "Overcast")
             .when(F.col("cover") == "OVX",   "Sky obscured")
             .otherwise(F.col("cover")))
        .withColumn("flight_category_name",
            F.when(F.col("fltCat") == "VFR",  "Visual Flight Rules")
             .when(F.col("fltCat") == "MVFR", "Marginal VFR")
             .when(F.col("fltCat") == "IFR",  "Instrument Flight Rules")
             .when(F.col("fltCat") == "LIFR", "Low Instrument Flight Rules")
             .otherwise(F.col("fltCat")))
        .withColumn("category_severity",
            F.when(F.col("fltCat") == "VFR",  1)
             .when(F.col("fltCat") == "MVFR", 2)
             .when(F.col("fltCat") == "IFR",  3)
             .when(F.col("fltCat") == "LIFR", 4)
             .otherwise(0))
        .withColumn("is_ifr",          F.col("fltCat").isin("IFR", "LIFR"))
        .withColumn("has_precipitation",
            F.when(F.col("wxString").isNotNull(),
                F.col("wxString").rlike("RA|SN|DZ|GR|GS|PL|SG|SH"))
             .otherwise(F.lit(False)))
        .withColumn("has_freezing",
            F.when(F.col("wxString").isNotNull(),
                F.col("wxString").rlike("FZ"))
             .otherwise(F.lit(False)))
        .withColumn("has_thunderstorm",
            F.when(F.col("wxString").isNotNull(),
                F.col("wxString").rlike("TS"))
             .otherwise(F.lit(False)))
        .withColumn("has_fog",
            F.when(F.col("wxString").isNotNull(),
                F.col("wxString").rlike("FG|BR|HZ"))
             .otherwise(F.lit(False)))
        # Deduplicate within this micro-batch
        .dropDuplicates(["icaoId", "obsTime"])
        .select(
            F.col("icaoId")         .alias("station_id"),
            F.col("name")           .alias("station_name"),
            "obs_time", "receipt_time",
            "temp_c", "dewpoint_c",
            "wind_dir_degrees", "wind_dir_variable",
            "wind_speed_kt", "wind_gust_kt",
            "visibility_sm", "visibility_unlimited",
            "altim_hpa",
            F.col("wxString")       .alias("wx_string"),
            "wx_string_decoded",
            "cover", "cover_description",
            "sky_cover", "cloud_base_ft",
            F.col("fltCat")         .alias("flight_category"),
            "flight_category_name", "category_severity",
            "is_ifr", "has_precipitation", "has_freezing",
            "has_thunderstorm", "has_fog",
            F.col("metarType")      .alias("metar_type"),
            "latitude", "longitude", "elev_m",
            F.col("rawOb")          .alias("raw_ob"),
            "_ingestion_timestamp"
        )
    )

    # MERGE: insert-only
    silver = DeltaTable.forName(spark, SILVER)
    (silver.alias("target")
        .merge(
            transformed.alias("source"),
            "target.station_id = source.station_id "
            "AND target.obs_time = source.obs_time"
        )
        .whenNotMatchedInsertAll()
        .execute()
    )


In [0]:
# ── 3. Stream: bronze to silver ────────────────────────────────────────
(
    spark.readStream
        .format("delta")
        .option("readChangeFeed", "true")
        .table(BRONZE)
    .writeStream
        .trigger(availableNow=True)
        .option("checkpointLocation", CHECKPOINT_PATH)
        .foreachBatch(transform_and_merge)
        .start()
        .awaitTermination()
)